In [2]:
import numpy as np
import json
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os

# ================= 配置 =================
EMB_FILE = "adni_kg_embeddings.npy"
ENT_FILE = "adni_kg_entity2id.json"

def autopsy_embeddings():
    print(f"🩺 开始对 {EMB_FILE} 进行尸检...\n")
    
    # 1. 加载文件
    if not os.path.exists(EMB_FILE) or not os.path.exists(ENT_FILE):
        print("❌ 找不到文件！请检查路径。")
        return

    emb = np.load(EMB_FILE)
    with open(ENT_FILE, 'r') as f:
        ent2id = json.load(f)
        
    print(f"📊 矩阵形状: {emb.shape}")
    print(f"📖 字典大小: {len(ent2id)}")
    
    if emb.shape[0] != len(ent2id):
        print(f"⚠️ 警告：矩阵行数 ({emb.shape[0]}) 与字典词数 ({len(ent2id)}) 不一致！这会导致索引错乱。")

    # 2. 检查数值分布 (Dead Neurons)
    print("\n[数值健康度检查]")
    print(f"   - Min: {emb.min():.6f}")
    print(f"   - Max: {emb.max():.6f}")
    print(f"   - Mean: {emb.mean():.6f}")
    print(f"   - Std (方差): {emb.std():.6f}")
    
    if emb.std() < 0.01:
        print("❌❌ 严重故障：方差极小！说明 Embedding 几乎没有区分度，模型坍缩了。")
        print("   -> 可能原因：学习率太高(梯度爆炸后全变0) 或 太低(根本没学动)。")
    elif emb.std() > 5.0:
        print("⚠️ 警告：方差过大，可能存在梯度爆炸。")
    else:
        print("✅ 方差正常。")

    # 3. 检查零向量 (Zero Vectors)
    norms = np.linalg.norm(emb, axis=1)
    zero_cnt = (norms == 0).sum()
    print(f"\n[零向量检查]")
    print(f"   - 零向量数量: {zero_cnt} / {len(emb)}")
    print(f"   - 占比: {zero_cnt / len(emb) * 100:.2f}%")
    
    if zero_cnt > 0:
        print("   -> ⚠️ 注意：部分节点没有学到任何东西。如果是少数孤立点可以接受。")

    # 4. 检查相似度坍缩 (Collapse Check)
    # 随机抽取 5 对不同的节点，计算它们的余弦相似度
    print("\n[区分度抽查 - 是否所有人都长得一样？]")
    
    # 专门提取 Patient 节点
    patient_ids = [v for k, v in ent2id.items() if "Patient" in k]
    if len(patient_ids) > 10:
        sample_idxs = np.random.choice(patient_ids, 5, replace=False)
        vecs = torch.tensor(emb[sample_idxs])
        
        # 计算两两相似度
        sim_matrix = F.cosine_similarity(vecs.unsqueeze(1), vecs.unsqueeze(0), dim=2)
        avg_sim = (sim_matrix.sum() - 5) / (5*4) # 去掉对角线
        
        print(f"   - 随机 5 个病人之间的平均相似度: {avg_sim:.4f}")
        
        if avg_sim > 0.95:
            print("❌❌ 致命错误：模型坍缩！(Model Collapse)")
            print("   说明：所有病人的向量几乎一模一样。模型无法区分谁是谁。")
            print("   -> 对策：增加 DistMult 的 Negative Sampling (负采样) 数量，或调大 Margin。")
        elif avg_sim < 0.1:
            print("⚠️ 警告：相似度过低，可能处于随机分布状态，没学到共性。")
        else:
            print("✅ 区分度良好：既有共性又有差异。")
    else:
        print("   (Patient 节点太少，跳过此项)")

    # 5. 检查语义关联 (Symptom vs PrimeKG)
    print("\n[语义关联抽查]")
    # 检查 'Symptom:npiq_ANX' 和 'PrimeKG:Anxiety' 的距离
    # 如果训练成功，它俩应该很近
    sym_key = "Symptom:npiq_ANX"
    pkg_key = "PrimeKG:Anxiety"
    
    if sym_key in ent2id and pkg_key in ent2id:
        v1 = torch.tensor(emb[ent2id[sym_key]])
        v2 = torch.tensor(emb[ent2id[pkg_key]])
        sim = F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()
        print(f"   - {sym_key} <--> {pkg_key} 相似度: {sim:.4f}")
        
        if sim < 0.3:
            print("⚠️ 警告：症状和对应的医学概念距离太远，知识没融合进去。")
    else:
        print("   (未找到测试用的 Anxiety 节点，跳过)")

autopsy_embeddings()

🩺 开始对 adni_kg_embeddings.npy 进行尸检...

📊 矩阵形状: (3370, 128)
📖 字典大小: 3370

[数值健康度检查]
   - Min: -1.615152
   - Max: 1.559714
   - Mean: 0.003401
   - Std (方差): 0.310563
✅ 方差正常。

[零向量检查]
   - 零向量数量: 0 / 3370
   - 占比: 0.00%

[区分度抽查 - 是否所有人都长得一样？]
   - 随机 5 个病人之间的平均相似度: 0.2948
✅ 区分度良好：既有共性又有差异。

[语义关联抽查]
   - Symptom:npiq_ANX <--> PrimeKG:Anxiety 相似度: -0.0324
⚠️ 警告：症状和对应的医学概念距离太远，知识没融合进去。


In [3]:
import pandas as pd
from tqdm import tqdm

# ================= 配置 =================
INPUT_TRIPLETS = 'adni_knowledge_triplets.csv'
OUTPUT_PATCHED = 'adni_knowledge_triplets_patched.csv'

def patch_graph():
    print(f"🔧 正在读取原始图谱: {INPUT_TRIPLETS} ...")
    df = pd.read_csv(INPUT_TRIPLETS)
    print(f"   - 原始边数: {len(df)}")
    
    # 1. 建立 "中间节点 -> PrimeKG节点" 的映射表
    # 寻找 (Symptom, mapped_to, PrimeKG) 的边
    symptom_map = {}
    mapped_rows = df[df['relation'] == 'mapped_to']
    
    for _, row in mapped_rows.iterrows():
        symptom_node = row['head']
        primekg_node = row['tail']
        symptom_map[symptom_node] = primekg_node
        
    print(f"   - 找到 {len(symptom_map)} 个症状映射桥梁。")
    
    # 2. 寻找病人到中间节点的边，并生成直连边
    # 寻找 (Patient, exhibits, Symptom)
    new_triplets = []
    exhibit_rows = df[df['relation'] == 'exhibits']
    
    count = 0
    for _, row in tqdm(exhibit_rows.iterrows(), total=len(exhibit_rows), desc="生成直连边"):
        patient = row['head']
        symptom = row['tail']
        
        # 如果这个症状能通向 PrimeKG
        if symptom in symptom_map:
            primekg_target = symptom_map[symptom]
            # ★★★ 生成直连边: Patient -> exhibits -> PrimeKG ★★★
            # 这样 DistMult 就能直接拉近病人和外部知识的距离
            new_triplets.append({
                'head': patient,
                'relation': 'exhibits', # 复用 exhibits 关系
                'tail': primekg_target
            })
            count += 1
            
    if count == 0:
        print("❌ 错误：没有生成任何新边！请检查 CSV 中是否有 exhibits 和 mapped_to 关系。")
        return

    # 3. 合并并保存
    df_new = pd.DataFrame(new_triplets)
    df_final = pd.concat([df, df_new], ignore_index=True)
    
    # 去重（防止重复添加）
    df_final.drop_duplicates(inplace=True)
    
    print(f"✅ 补丁完成！")
    print(f"   - 新增直连边: {count}")
    print(f"   - 总边数: {len(df_final)}")
    
    df_final.to_csv(OUTPUT_PATCHED, index=False)
    print(f"💾 已保存修复后的图谱: {OUTPUT_PATCHED}")

patch_graph()

🔧 正在读取原始图谱: adni_knowledge_triplets.csv ...
   - 原始边数: 21888
   - 找到 10 个症状映射桥梁。


生成直连边: 100%|█████████████████████████████████████████████████████████████████| 921/921 [00:00<00:00, 10890.76it/s]

✅ 补丁完成！
   - 新增直连边: 921
   - 总边数: 22809
💾 已保存修复后的图谱: adni_knowledge_triplets_patched.csv
